# LIME / SHAP / Optuna — CIC vs VM Feature Analysis

Compares feature importance between the **CIC-IDS2017** baseline model and
a freshly **Optuna-tuned model on VM-generated flows** using:

| Tool | Role |
|------|------|
| **Optuna** | Tune XGBoost on VM data (TPE, 40 trials) |
| **SHAP** | Global TreeExplainer importance — CIC baseline vs VM model |
| **LIME** | Aggregated instance-level importance — CIC baseline vs VM model |

### Preprocessing contract (matches existing notebooks)
Both datasets go through the exact same pipeline already used in
`LeanNotebook.ipynb` and `LeanNotebookWithGT.ipynb`:
1. Load CSV → strip column whitespace → replace `inf`
2. Apply `rename_map.pkl`
3. Duplicate `Fwd Header Length` → `Fwd Header Length.1`
4. Select `feature_columns` from `preprocessing_info.pkl`
5. **No scaling** — XGBoost is tree-based, scaling not needed or used

> Run cells top-to-bottom on first execution.  
> After Optuna completes you can freely re-run SHAP/LIME cells without re-tuning.

## 0 · Imports & Configuration

In [ ]:
import warnings, pickle, json, os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing  import LabelEncoder
from sklearn.metrics        import classification_report, f1_score

import xgboost as xgb
import optuna
import shap
import lime
import lime.lime_tabular

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
shap.initjs()

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Imports OK')

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────
# CIC data = the original CIC-IDS2017 labeled CSV used to train the baseline
# VM data  = your VM-generated labeled flows from pcap_to_labeled_flows.py
CIC_CSV        = 'data/cic_flows.csv'         # adjust if named differently
VM_CSV         = 'data/flows.csv'              # your VM labeled flows
MODEL_PATH     = 'models/baseline_xgboost.json'
ARTIFACTS_PATH = 'models/preprocessing_info.pkl'
RENAME_PATH    = 'src/preprocessing/rename_map.pkl'
OUT_DIR        = Path('results/lime_shap')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Analysis settings ─────────────────────────────────────────────────
N_TRIALS  = 40    # Optuna trials for VM model (raise for better tuning)
TOP_N     = 20    # Features shown in plots and comparison table
LIME_SAMP = 300   # Test instances explained by LIME  (lower = faster)
SHAP_CAP  = 1000  # Max rows passed to SHAP (memory guard)

# ── Colour palette ─────────────────────────────────────────────────────
CIC_COL  = '#3A86FF'
VM_COL   = '#FF006E'
BOTH_COL = '#8338EC'
print(' Config set')

## 1 · Load Shared Artefacts

The `preprocessing_info.pkl` contains:
- `feature_columns` — the exact ordered feature list the baseline model expects
- `target_encoder` — `LabelEncoder` mapping string labels ↔ integers
- `categorical_encoders` / `categorical_columns` — encoders for IP, protocol, etc.

The `rename_map.pkl` maps raw CICFlowMeter column names to the canonical names
used by the model (same step used in every other notebook).

In [ ]:
# Load preprocessing artefacts
with open(ARTIFACTS_PATH, 'rb') as f:
    artifacts = pickle.load(f)

le_target    = artifacts['target_encoder']    # LabelEncoder
feature_cols = artifacts['feature_columns']   # ordered list
cat_encoders = artifacts.get('categorical_encoders', {})
cat_cols     = artifacts.get('categorical_columns', [])

with open(RENAME_PATH, 'rb') as f:
    rename_map = pickle.load(f)

# The baseline XGBoost model (trained on CIC-IDS2017)
baseline_model = xgb.XGBClassifier()
baseline_model.load_model(MODEL_PATH)

print(f'Classes       : {list(le_target.classes_)}')
print(f'Feature count : {len(feature_cols)}')
print(f'Cat columns   : {cat_cols}')

## 2 · Preprocessing Function

Single reusable function that mirrors `LeanNotebookWithGT.ipynb` exactly:
rename → header fix → categorical encode → select features → drop NaN rows.

In [ ]:
def preprocess(csv_path: str, label_col: str = 'Label'):
    """
    Load and preprocess a labeled flows CSV.
    Returns (X_df, y_encoded, y_raw_series) using the shared feature_cols.
    """
    df = pd.read_csv(csv_path, low_memory=False)
    df.columns = df.columns.str.strip()
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Step 1 — rename raw CICFlowMeter columns
    df.rename(columns=rename_map, inplace=True)

    # Step 2 — CICIDS2017 duplicate header fix (model expects this column)
    if 'Fwd Header Length' in df.columns:
        df['Fwd Header Length.1'] = df['Fwd Header Length']

    # Step 3 — encode categorical columns (IP, protocol, etc.)
    for col in cat_cols:
        if col in df.columns and col in cat_encoders:
            enc = cat_encoders[col]
            # Handle unseen labels gracefully
            known = set(enc.classes_)
            df[col] = df[col].astype(str).apply(
                lambda x: x if x in known else enc.classes_[0]
            )
            df[col] = enc.transform(df[col])

    # Step 4 — extract label BEFORE selecting feature cols
    y_raw = df[label_col].astype(str).str.strip() if label_col in df.columns \
            else pd.Series(['UNKNOWN'] * len(df))

    # Step 5 — select & clean features
    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        print(f'  [!] Adding {len(missing)} missing columns as 0: {missing[:5]}...')
        for c in missing:
            df[c] = 0

    X = df[feature_cols].copy()
    X.fillna(X.median(numeric_only=True), inplace=True)

    # Encode labels to integers
    known_classes = set(le_target.classes_)
    y_raw_clean = y_raw.apply(lambda x: x if x in known_classes else le_target.classes_[0])
    y_enc = le_target.transform(y_raw_clean)

    print(f'  Loaded  : {csv_path}')
    print(f'  Shape   : {X.shape}')
    print(f'  Classes : {y_raw.value_counts().to_dict()}')
    return X, y_enc, y_raw


print('preprocess() defined')

## 3 · Load & Inspect Both Datasets

In [ ]:
print('── CIC Dataset ─────────────────────────────────')
X_cic, y_cic, y_cic_raw = preprocess(CIC_CSV)

print('\n── VM Dataset ──────────────────────────────────')
X_vm,  y_vm,  y_vm_raw  = preprocess(VM_CSV)

In [ ]:
# Side-by-side class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, y_raw, title, color in [
    (axes[0], y_cic_raw, 'CIC-IDS2017', CIC_COL),
    (axes[1], y_vm_raw,  'VM Traffic',  VM_COL),
]:
    counts = y_raw.value_counts()
    ax.barh(counts.index, counts.values, color=color, edgecolor='white', linewidth=0.4)
    ax.set_title(f'Class Distribution — {title}', fontweight='bold')
    ax.set_xlabel('Flow count')
plt.tight_layout()
plt.savefig(OUT_DIR / 'class_distribution.png', bbox_inches='tight')
plt.show()

## 4 · Train / Test Split

No scaling step — XGBoost is tree-based and the baseline model was trained
without a scaler (confirmed: `preprocessing_info.pkl` contains no scaler key).

In [ ]:
np.random.seed(42)

Xtr_cic, Xte_cic, ytr_cic, yte_cic = train_test_split(
    X_cic, y_cic, test_size=0.2, random_state=42, stratify=y_cic)

Xtr_vm, Xte_vm, ytr_vm, yte_vm = train_test_split(
    X_vm, y_vm, test_size=0.2, random_state=42, stratify=y_vm)

n_classes = len(le_target.classes_)
class_names = list(le_target.classes_)

print(f'CIC  train={len(Xtr_cic):,}  test={len(Xte_cic):,}')
print(f'VM   train={len(Xtr_vm):,}  test={len(Xte_vm):,}')
print(f'Classes ({n_classes}): {class_names}')

## 5 · CIC Side — Evaluate Existing Baseline Model

Rather than re-tuning on CIC data (which could take hours on the full dataset),
we use the pre-trained `baseline_xgboost.json` directly for SHAP/LIME.
This is the model already trained on CIC-IDS2017, so its SHAP values
*are* the CIC feature importance picture.

In [ ]:
# Evaluate baseline on CIC test split
yte_cic_pred = baseline_model.predict(Xte_cic)
report_cic   = classification_report(
    yte_cic, yte_cic_pred, target_names=class_names, zero_division=0)
print('── CIC Baseline Classification Report ──')
print(report_cic)

## 6 · VM Side — Optuna Hyperparameter Tuning

We tune a fresh XGBoost on the VM data so the comparison is fair:
CIC model = optimised for CIC, VM model = optimised for VM.

If the best hyperparameters diverge significantly from what worked for CIC,
that itself is a domain-mismatch signal.

> ⏱️ ~3–8 min depending on VM dataset size. Reduce `N_TRIALS` at the top for a quicker run.

In [ ]:
def make_objective(X_tr, y_tr, X_va, y_va):
    def objective(trial):
        params = dict(
            n_estimators     = trial.suggest_int  ('n_estimators',    100, 600, step=50),
            max_depth        = trial.suggest_int  ('max_depth',         3,  10),
            learning_rate    = trial.suggest_float('learning_rate',  1e-3, 0.3, log=True),
            subsample        = trial.suggest_float('subsample',       0.5, 1.0),
            colsample_bytree = trial.suggest_float('colsample_bytree',0.4, 1.0),
            min_child_weight = trial.suggest_int  ('min_child_weight',  1,  10),
            reg_alpha        = trial.suggest_float('reg_alpha',      1e-4,10.0, log=True),
            reg_lambda       = trial.suggest_float('reg_lambda',     1e-4,10.0, log=True),
            objective        = 'multi:softprob' if n_classes > 2 else 'binary:logistic',
            eval_metric      = 'mlogloss'       if n_classes > 2 else 'logloss',
            use_label_encoder= False,
            verbosity        = 0, tree_method='hist', random_state=42,
        )
        if n_classes > 2:
            params['num_class'] = n_classes
        m = xgb.XGBClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              verbose=False, early_stopping_rounds=20)
        return f1_score(y_va, m.predict(X_va), average='weighted', zero_division=0)
    return objective


# 80/20 sub-split of VM training data for Optuna validation
Xtr2_vm, Xva_vm, ytr2_vm, yva_vm = train_test_split(
    Xtr_vm, ytr_vm, test_size=0.2, random_state=42, stratify=ytr_vm)

print(f'[VM] Running {N_TRIALS} Optuna trials ...')
study_vm = optuna.create_study(direction='maximize',
                               sampler=optuna.samplers.TPESampler(seed=42))
study_vm.optimize(
    make_objective(Xtr2_vm, ytr2_vm, Xva_vm, yva_vm),
    n_trials=N_TRIALS, show_progress_bar=True)

best_vm = study_vm.best_params
print(f'\n VM best weighted-F1: {study_vm.best_value:.4f}')
print(json.dumps(best_vm, indent=2))

In [ ]:
# Optuna visualisations
from optuna.visualization.matplotlib import (
    plot_optimization_history, plot_param_importances)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plt.sca(axes[0]); plot_optimization_history(study_vm, target_name='Weighted F1')
axes[0].set_title('VM — Optimisation History', fontweight='bold')
plt.sca(axes[1]); plot_param_importances(study_vm)
axes[1].set_title('VM — Hyperparameter Importance', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'optuna_vm.png', bbox_inches='tight')
plt.show()

## 7 · Train Final VM Model

In [ ]:
p = dict(best_vm,
         objective        = 'multi:softprob' if n_classes > 2 else 'binary:logistic',
         eval_metric      = 'mlogloss'       if n_classes > 2 else 'logloss',
         use_label_encoder= False,
         verbosity=0, tree_method='hist', random_state=42)
if n_classes > 2:
    p['num_class'] = n_classes

vm_model = xgb.XGBClassifier(**p)
vm_model.fit(Xtr_vm, ytr_vm, verbose=False)

report_vm = classification_report(
    yte_vm, vm_model.predict(Xte_vm), target_names=class_names, zero_division=0)
print('── VM Tuned Model Classification Report ──')
print(report_vm)

## 8 · SHAP Analysis

**How to read these plots:**
- **Bar chart** — mean `|SHAP value|` per feature across all test flows (global importance).
- **Beeswarm** — each dot = one flow. Red = high feature value, blue = low.
  Dot pushed right → feature **increased** the attack prediction score.

For multi-class output, SHAP values are averaged across all classes
so the bar chart gives a single global view.

In [ ]:
def run_shap(model, X_test_df, label, color):
    X_arr = X_test_df.values
    cap   = min(SHAP_CAP, len(X_arr))
    bg    = X_arr[np.random.choice(len(X_arr), min(200, len(X_arr)), replace=False)]

    explainer = shap.TreeExplainer(model, data=bg,
                                   feature_perturbation='interventional')
    sv = explainer(X_arr[:cap])

    vals = sv.values
    if vals.ndim == 3:             # multi-class
        vals_abs = np.abs(vals).mean(axis=2)
    else:
        vals_abs = np.abs(vals)

    mean_shap = pd.Series(vals_abs.mean(axis=0),
                          index=feature_cols).sort_values(ascending=False)

    # Bar chart
    fig, ax = plt.subplots(figsize=(9, 6))
    top = mean_shap.head(TOP_N)
    ax.barh(top.index[::-1], top.values[::-1], color=color,
            edgecolor='white', linewidth=0.4)
    ax.set_xlabel('Mean |SHAP value|')
    ax.set_title(f'SHAP Feature Importance — {label}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'shap_bar_{label.lower()}.png', bbox_inches='tight')
    plt.show()

    # Beeswarm (top-N features)
    top_idx = [feature_cols.index(f) for f in mean_shap.head(TOP_N).index
               if f in feature_cols]
    base = sv.base_values if vals.ndim == 2 else sv.base_values[:, 0]
    sv_top = shap.Explanation(
        values       = vals_abs[:, top_idx],
        base_values  = base,
        data         = X_arr[:cap][:, top_idx],
        feature_names= mean_shap.head(TOP_N).index.tolist(),
    )
    plt.figure(figsize=(10, 7))
    shap.plots.beeswarm(sv_top, max_display=TOP_N, show=False)
    plt.title(f'SHAP Beeswarm — {label}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'shap_beeswarm_{label.lower()}.png', bbox_inches='tight')
    plt.show()

    return mean_shap


print('[SHAP] CIC baseline ...')
shap_cic = run_shap(baseline_model, Xte_cic, 'CIC', CIC_COL)
print('[SHAP] VM tuned model ...')
shap_vm  = run_shap(vm_model,       Xte_vm,  'VM',  VM_COL)

## 9 · LIME Analysis

LIME fits a local linear model around each individual prediction.  
We explain `LIME_SAMP` random test flows and aggregate absolute weights
to build a global importance picture.

> ⏱️ Slowest cell. Reduce `LIME_SAMP` in Cell 0 config if needed.

In [ ]:
def run_lime(model, X_train_df, X_test_df, label, color):
    X_tr  = X_train_df.values
    X_te  = X_test_df.values
    explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data        = X_tr,
        feature_names        = feature_cols,
        class_names          = class_names,
        mode                 = 'classification',
        random_state         = 42,
        discretize_continuous= True,
    )

    idx_sample = np.random.choice(len(X_te), min(LIME_SAMP, len(X_te)), replace=False)
    agg = np.zeros(len(feature_cols))

    for i, idx in enumerate(idx_sample):
        if i % 50 == 0:
            print(f'  {label}: explained {i}/{len(idx_sample)}', end='\r')
        exp = explainer.explain_instance(
            X_te[idx], model.predict_proba,
            num_features=len(feature_cols),
            labels=list(range(n_classes)),
        )
        for cls_idx in range(n_classes):
            for feat_idx, weight in exp.local_exp.get(cls_idx, []):
                agg[feat_idx] += abs(weight)

    agg /= (len(idx_sample) * n_classes)
    lime_imp = pd.Series(agg, index=feature_cols).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(9, 6))
    top = lime_imp.head(TOP_N)
    ax.barh(top.index[::-1], top.values[::-1], color=color,
            edgecolor='white', linewidth=0.4)
    ax.set_xlabel('Mean |LIME weight|')
    ax.set_title(f'LIME Feature Importance — {label}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'lime_bar_{label.lower()}.png', bbox_inches='tight')
    plt.show()

    print(f'\n  {label} top-5: {lime_imp.head(5).to_dict()}')
    return lime_imp


print('[LIME] CIC baseline ...')
lime_cic = run_lime(baseline_model, Xtr_cic, Xte_cic, 'CIC', CIC_COL)
print('[LIME] VM tuned model ...')
lime_vm  = run_lime(vm_model,       Xtr_vm,  Xte_vm,  'VM',  VM_COL)

## 10 · Feature Comparison Table

| Column | Meaning |
|--------|---------|
| `CIC_SHAP` / `VM_SHAP` | Normalised SHAP importance [0–1] |
| `CIC_LIME` / `VM_LIME` | Normalised LIME importance [0–1] |
| `Composite` | Mean of all four — primary sort key |
| `Rank_Delta` | `|CIC_avg_rank − VM_avg_rank|` — disagreement indicator |
| `Presence` | `BOTH` / `CIC-ONLY` / `VM-ONLY` — where does this feature matter? |
| `Domain_Gap` | `HIGH` (Δ>5) / `MED` (Δ>2) — transfer-risk flag |

In [ ]:
def norm(s):
    r = s.max() - s.min()
    return (s - s.min()) / r if r > 0 else s / s.max()

tbl = pd.concat([
    norm(shap_cic).rename('CIC_SHAP'),
    norm(shap_vm ).rename('VM_SHAP'),
    norm(lime_cic).rename('CIC_LIME'),
    norm(lime_vm ).rename('VM_LIME'),
], axis=1).fillna(0.0)

tbl['Composite']     = tbl.mean(axis=1)
tbl.sort_values('Composite', ascending=False, inplace=True)

tbl['CIC_SHAP_rank'] = shap_cic.rank(ascending=False, method='min').astype(int)
tbl['VM_SHAP_rank']  = shap_vm .rank(ascending=False, method='min').astype(int)
tbl['CIC_LIME_rank'] = lime_cic.rank(ascending=False, method='min').astype(int)
tbl['VM_LIME_rank']  = lime_vm .rank(ascending=False, method='min').astype(int)
tbl['CIC_AvgRank']   = (tbl['CIC_SHAP_rank'] + tbl['CIC_LIME_rank']) / 2
tbl['VM_AvgRank']    = (tbl['VM_SHAP_rank']  + tbl['VM_LIME_rank'])  / 2
tbl['Rank_Delta']    = (tbl['CIC_AvgRank'] - tbl['VM_AvgRank']).abs().round(1)

cic_top = set(shap_cic.head(TOP_N).index) | set(lime_cic.head(TOP_N).index)
vm_top  = set(shap_vm .head(TOP_N).index) | set(lime_vm .head(TOP_N).index)
def tag(f):
    ic, iv = f in cic_top, f in vm_top
    return 'BOTH' if ic and iv else ('CIC-ONLY' if ic else ('VM-ONLY' if iv else 'NEITHER'))
tbl['Presence']    = tbl.index.map(tag)
tbl['Domain_Gap']  = tbl.apply(
    lambda r: 'HIGH' if r['Presence']=='BOTH' and r['Rank_Delta']>5
         else ('MED'  if r['Presence']=='BOTH' and r['Rank_Delta']>2 else '—'), axis=1)

tbl.to_csv(OUT_DIR / 'feature_comparison_table.csv')
print(f' Saved → {OUT_DIR}/feature_comparison_table.csv')

In [ ]:
# Styled display — green = shared, blue = CIC-only, red = VM-only
display_cols = ['CIC_SHAP','VM_SHAP','CIC_LIME','VM_LIME',
                'Composite','Rank_Delta','Presence','Domain_Gap']

def _col_presence(v):
    return {'BOTH':     'background-color:#d4edda;color:#155724',
            'CIC-ONLY': 'background-color:#cce5ff;color:#004085',
            'VM-ONLY':  'background-color:#f8d7da;color:#721c24',
            }.get(v, '')

def _col_gap(v):
    return {'HIGH': 'background-color:#f8d7da;font-weight:bold',
            'MED':  'background-color:#fff3cd'}.get(v, '')

(tbl.head(TOP_N)[display_cols]
 .style
 .background_gradient(subset=['CIC_SHAP','VM_SHAP','CIC_LIME','VM_LIME','Composite'],
                      cmap='YlGn')
 .background_gradient(subset=['Rank_Delta'], cmap='Reds')
 .applymap(_col_presence, subset=['Presence'])
 .applymap(_col_gap,      subset=['Domain_Gap'])
 .format('{:.3f}', subset=['CIC_SHAP','VM_SHAP','CIC_LIME','VM_LIME','Composite','Rank_Delta']))

## 11 · Cross-Dataset Visualisations

### 11a · Importance Heatmap — CIC vs VM

In [ ]:
top    = tbl.head(TOP_N)
heat_d = top[['CIC_SHAP','VM_SHAP','CIC_LIME','VM_LIME']].T

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(heat_d, annot=True, fmt='.2f', linewidths=0.4, cmap='RdYlGn', ax=ax,
            cbar_kws={'label': 'Normalised Importance'},
            xticklabels=top.index,
            yticklabels=['CIC SHAP','VM SHAP','CIC LIME','VM LIME'])
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_title(f'Feature Importance Heatmap — CIC vs VM (Top {TOP_N})',
             fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig(OUT_DIR / 'heatmap_cic_vs_vm.png', bbox_inches='tight')
plt.show()

### 11b · Domain Gap — Rank Delta

In [ ]:
shared = tbl[tbl['Presence']=='BOTH'].head(TOP_N).sort_values('Rank_Delta', ascending=False)
colors = ['#d62728' if g=='HIGH' else '#ff7f0e' if g=='MED' else '#2ca02c'
          for g in shared['Domain_Gap']]

fig, ax = plt.subplots(figsize=(10, max(4, len(shared)*0.42)))
ax.barh(shared.index, shared['Rank_Delta'], color=colors, edgecolor='white', linewidth=0.3)
ax.set_xlabel('|CIC_AvgRank − VM_AvgRank|')
ax.set_title('Domain Gap — Shared Features', fontweight='bold')
ax.legend(handles=[
    mpatches.Patch(color='#d62728', label='HIGH gap (Δ > 5)'),
    mpatches.Patch(color='#ff7f0e', label='MED  gap (Δ > 2)'),
    mpatches.Patch(color='#2ca02c', label='LOW  gap'),
], fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / 'rank_delta.png', bbox_inches='tight')
plt.show()

### 11c · Feature Overlap Count

In [ ]:
cats   = ['BOTH','CIC-ONLY','VM-ONLY','NEITHER']
counts = tbl['Presence'].value_counts()
vals   = [counts.get(c, 0) for c in cats]
colors = [BOTH_COL, CIC_COL, VM_COL, '#aaaaaa']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(cats, vals, color=colors, edgecolor='white')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            str(v), ha='center', fontweight='bold')
ax.set_ylabel('Feature count')
ax.set_title('Feature Overlap: CIC vs VM', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'feature_overlap.png', bbox_inches='tight')
plt.show()

### 11d · SHAP Divergence Scatter

In [ ]:
# Points ABOVE diagonal → more important in VM than CIC
# Points BELOW diagonal → more important in CIC (possible lab artefact)
fig, ax = plt.subplots(figsize=(8, 7))
cfg = {'BOTH':    (BOTH_COL,80,'o','Shared'),
       'CIC-ONLY':(CIC_COL, 60,'s','CIC-only'),
       'VM-ONLY': (VM_COL,  60,'^','VM-only'),
       'NEITHER': ('#bbb',  30,'.','Neither')}
for pres,(col,sz,mk,lbl) in cfg.items():
    sub = tbl[tbl['Presence']==pres]
    ax.scatter(sub['CIC_SHAP'], sub['VM_SHAP'], c=col, s=sz, marker=mk,
               alpha=0.75, label=lbl, edgecolors='white', linewidth=0.4)
for feat in tbl[tbl['Presence']=='BOTH'].head(8).index:
    ax.annotate(feat, (tbl.loc[feat,'CIC_SHAP'], tbl.loc[feat,'VM_SHAP']),
                fontsize=6.5, xytext=(3,3), textcoords='offset points')
lim = max(tbl['CIC_SHAP'].max(), tbl['VM_SHAP'].max()) * 1.05
ax.plot([0,lim],[0,lim],'k--',lw=0.8,alpha=0.4,label='y = x')
ax.set_xlabel('CIC SHAP (normalised)')
ax.set_ylabel('VM  SHAP (normalised)')
ax.set_title('SHAP Divergence — CIC vs VM', fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / 'shap_divergence.png', bbox_inches='tight')
plt.show()

## 12 · Summary

In [ ]:
both_feats = tbl[tbl['Presence']=='BOTH']
cic_only   = tbl[tbl['Presence']=='CIC-ONLY']
vm_only    = tbl[tbl['Presence']=='VM-ONLY']
high_gap   = tbl[tbl['Domain_Gap']=='HIGH']

summary = f"""
{'='*65}
 ANALYSIS SUMMARY — CIC Baseline vs VM Tuned Model
{'='*65}

VM Optuna Best Params
{json.dumps(best_vm, indent=2)}

Feature Overlap  (top-{TOP_N} by Composite Importance)
  Shared (BOTH)   : {len(both_feats)}
  CIC-exclusive   : {len(cic_only)}  {list(cic_only.index[:5])}
  VM-exclusive    : {len(vm_only)}   {list(vm_only.index[:5])}

High Domain-Gap Features  (|ΔRank| > 5)
{'  None detected' if len(high_gap)==0 else chr(10).join(
    f'  {f:<40} CIC={r.CIC_AvgRank:.1f}  VM={r.VM_AvgRank:.1f}  Δ={r.Rank_Delta:.1f}'
    for f, r in high_gap.iterrows())}

What this means
  BOTH features     → Robust indicators. Reliable across CIC lab data
                      and real VM traffic. Keep these in production.
  CIC-ONLY features → Likely lab artefacts (scripted timing, fixed
                      packet sizes). May not generalise to real traffic.
  VM-ONLY features  → Real-world network quirks absent in CIC-IDS2017.
                      Important for keeping the detector current.
  HIGH Domain_Gap   → Feature present in both but ranked very differently.
                      Context-dependent or noisy — consider engineering.
{'='*65}
"""

print(summary)
with open(OUT_DIR / 'analysis_summary.txt', 'w') as fh:
    fh.write(summary)
print(f' Saved → {OUT_DIR}/analysis_summary.txt')